In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Conv2DTranspose, Concatenate, add, BatchNormalization, Activation
import os
import glob
import tifffile as tiff
# import numpy as np
# import os
# import matplotlib.pyplot as plt
# from sklearn.metrics import precision_score, recall_score, f1_score
from PIL import Image
import rasterio
from sklearn.model_selection import train_test_split;
import re
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

tf.keras.backend.clear_session()

2026-04-02 03:14:49.504832: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775099689.702926      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775099689.757140      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775099690.185835      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775099690.185874      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775099690.185877      55 computation_placer.cc:177] computation placer alr

In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
def conv_block(x, filters, batchnorm=True):
    conv1 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm is True:
        conv1 = BatchNormalization(axis=3)(conv1)
    conv1 = Activation('relu')(conv1)    
    conv2 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(conv1)
    if batchnorm is True:
        conv2 = BatchNormalization(axis=3)(conv2)
    conv2 = Activation("relu")(conv2)

    return conv2

In [4]:
def residual_conv_block(x, filters, batchnorm=True):
    conv1 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm is True:
        conv1 = BatchNormalization(axis=3)(conv1)
    conv1 = Activation('relu')(conv1)    
    conv2 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(conv1)
    if batchnorm is True:
        conv2 = BatchNormalization(axis=3)(conv2)
    conv2 = Activation("relu")(conv2)
        
    #skip connection    
    shortcut = Conv2D(filters, kernel_size=(1, 1), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm is True:
        shortcut = BatchNormalization(axis=3)(shortcut)
    shortcut = Activation("relu")(shortcut)
    respath = add([shortcut, conv2])       
    return respath

In [5]:
def dense_block(inputs, num_filters):
    conv1 = conv_block(inputs, num_filters)
    concat = Concatenate()([inputs, conv1])
    return concat

In [6]:
def residual_unet(input_shape, base_filters=64):
    inputs = Input(input_shape)
    
    # Encoder
    conv1 = residual_conv_block(inputs, base_filters)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
    conv2 = residual_conv_block(pool1, base_filters * 2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
    conv3 = residual_conv_block(pool2, base_filters * 4)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)
    conv4 = residual_conv_block(pool3, base_filters * 8)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)
    
    # Bottleneck
    conv5 = Conv2D(base_filters * 16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(pool4)
    conv5 = Conv2D(base_filters * 16, (3, 3), kernel_initializer='he_normal', padding='same')(conv5)
    conv5 = Activation('relu')(conv5)
    drop5 = Dropout(0.5)(conv5)
    
    # Decoder
    up6 = Conv2DTranspose(base_filters * 8, (2, 2), strides=(2, 2), padding='same')(drop5)
    up6 = Concatenate()([up6, conv4])
    conv6 = residual_conv_block(up6, base_filters * 8)
    up7 = Conv2DTranspose(base_filters * 4, (2, 2), strides=(2, 2), padding='same')(conv6)
    up7 = Concatenate()([up7, conv3])
    conv7 = residual_conv_block(up7, base_filters * 4)
    up8 = Conv2DTranspose(base_filters * 2, (2, 2), strides=(2, 2), padding='same')(conv7)
    up8 = Concatenate()([up8, conv2])
    conv8 = residual_conv_block(up8, base_filters * 2)
    up9 = Conv2DTranspose(base_filters, (2, 2), strides=(2, 2), padding='same')(conv8)
    up9 = Concatenate()([up9, conv1])
    conv9 = residual_conv_block(up9, base_filters)
    
    # Output
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(conv9)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [7]:
def dense_unet(input_shape, base_filters=64):
    inputs = Input(input_shape)

    # Encoder (dense blocks with dropout)
    conv1 = dense_block(inputs, base_filters)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = dense_block(pool1, base_filters * 2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = dense_block(pool2, base_filters * 4)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = dense_block(pool3, base_filters * 8)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)

    # Bottleneck (two convs + dropout)
    conv5 = Conv2D(base_filters * 16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(pool4)
    conv5 = Conv2D(base_filters * 16, (3, 3), kernel_initializer='he_normal', padding='same')(conv5)
    conv5 = Activation('relu')(conv5)
    drop5 = Dropout(0.5)(conv5)  # core stochastic layer

    # Decoder (transpose convs + residual blocks + dropout)
    up6 = Conv2DTranspose(base_filters * 8, (2, 2), strides=(2, 2), padding='same')(drop5)
    up6 = Concatenate()([up6, conv4])
    conv6 = dense_block(up6, base_filters * 8)

    up7 = Conv2DTranspose(base_filters * 4, (2, 2), strides=(2, 2), padding='same')(conv6)
    up7 = Concatenate()([up7, conv3])
    conv7 = dense_block(up7, base_filters * 4)

    up8 = Conv2DTranspose(base_filters * 2, (2, 2), strides=(2, 2), padding='same')(conv7)
    up8 = Concatenate()([up8, conv2])
    conv8 = dense_block(up8, base_filters * 2)

    up9 = Conv2DTranspose(base_filters, (2, 2), strides=(2, 2), padding='same')(conv8)
    up9 = Concatenate()([up9, conv1])
    conv9 = dense_block(up9, base_filters)

    # Output - binary segmentation (sigmoid)
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(conv9)

    model = Model(inputs=inputs, outputs=outputs)
    return model

In [8]:
def dense_unet_plus_plus(input_shape, base_filters=64):
    inputs = Input(input_shape)

    # ======================
    # Encoder (X_{i,0})
    # ======================
    x0_0 = dense_block(inputs, base_filters)
    p0 = MaxPooling2D((2, 2))(x0_0)

    x1_0 = dense_block(p0, base_filters * 2)
    p1 = MaxPooling2D((2, 2))(x1_0)

    x2_0 = dense_block(p1, base_filters * 4)
    p2 = MaxPooling2D((2, 2))(x2_0)

    x3_0 = dense_block(p2, base_filters * 8)
    p3 = MaxPooling2D((2, 2))(x3_0)

    # ======================
    # Bottleneck
    # ======================
    x4_0 = Conv2D(base_filters * 16, (3, 3), padding='same',
                  kernel_initializer='he_normal', activation='relu')(p3)
    x4_0 = Conv2D(base_filters * 16, (3, 3), padding='same',
                  kernel_initializer='he_normal', activation='relu')(x4_0)
    x4_0 = Dropout(0.5)(x4_0)

    # ======================
    # Decoder: Level 1
    # ======================
    x3_1 = dense_block(
        Concatenate()([
            x3_0,
            Conv2DTranspose(base_filters * 8, (2, 2), strides=(2, 2), padding='same')(x4_0)
        ]),
        base_filters * 8
    )

    x2_1 = dense_block(
        Concatenate()([
            x2_0,
            Conv2DTranspose(base_filters * 4, (2, 2), strides=(2, 2), padding='same')(x3_0)
        ]),
        base_filters * 4
    )

    x1_1 = dense_block(
        Concatenate()([
            x1_0,
            Conv2DTranspose(base_filters * 2, (2, 2), strides=(2, 2), padding='same')(x2_0)
        ]),
        base_filters * 2
    )

    x0_1 = dense_block(
        Concatenate()([
            x0_0,
            Conv2DTranspose(base_filters, (2, 2), strides=(2, 2), padding='same')(x1_0)
        ]),
        base_filters
    )

    # ======================
    # Decoder: Level 2
    # ======================
    x2_2 = dense_block(
        Concatenate()([
            x2_0, x2_1,
            Conv2DTranspose(base_filters * 4, (2, 2), strides=(2, 2), padding='same')(x3_1)
        ]),
        base_filters * 4
    )

    x1_2 = dense_block(
        Concatenate()([
            x1_0, x1_1,
            Conv2DTranspose(base_filters * 2, (2, 2), strides=(2, 2), padding='same')(x2_1)
        ]),
        base_filters * 2
    )

    x0_2 = dense_block(
        Concatenate()([
            x0_0, x0_1,
            Conv2DTranspose(base_filters, (2, 2), strides=(2, 2), padding='same')(x1_1)
        ]),
        base_filters
    )

    # ======================
    # Decoder: Level 3
    # ======================
    x1_3 = dense_block(
        Concatenate()([
            x1_0, x1_1, x1_2,
            Conv2DTranspose(base_filters * 2, (2, 2), strides=(2, 2), padding='same')(x2_2)
        ]),
        base_filters * 2
    )

    x0_3 = dense_block(
        Concatenate()([
            x0_0, x0_1, x0_2,
            Conv2DTranspose(base_filters, (2, 2), strides=(2, 2), padding='same')(x1_2)
        ]),
        base_filters
    )

    # ======================
    # Decoder: Level 4
    # ======================
    x0_4 = dense_block(
        Concatenate()([
            x0_0, x0_1, x0_2, x0_3,
            Conv2DTranspose(base_filters, (2, 2), strides=(2, 2), padding='same')(x1_3)
        ]),
        base_filters
    )
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(x0_4)

    model = Model(inputs, outputs)
    return model


In [9]:
def load_data_new(image_dir, mask_dir):
    import os
    import numpy as np
    import rasterio

    # Check directories
    if not os.path.exists(image_dir):
        print(f"Image directory {image_dir} does not exist.")
        return None, None

    if not os.path.exists(mask_dir):
        print(f"Mask directory {mask_dir} does not exist.")
        return None, None

    # Get all image files
    image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.tif')])

    if len(image_files) == 0:
        print(f"No .tif images found in {image_dir}")
        return None, None

    images = []
    masks = []

    for file in image_files:
        img_path = os.path.join(image_dir, file)
        mask_path = os.path.join(mask_dir, file)

        # Ensure matching mask exists
        if not os.path.exists(mask_path):
            print(f"Mask missing for {file}")
            continue

        # ---- READ IMAGE (MULTI-BAND) ----
        with rasterio.open(img_path) as src:
            img = src.read()   # (C, H, W)
            img = np.transpose(img, (1, 2, 0))  # → (H, W, C)

        # ---- READ MASK ----
        with rasterio.open(mask_path) as src:
            mask = src.read(1)  # (H, W)

        # ---- NORMALIZE IMAGE (PER CHANNEL) ----
        img = img.astype(np.float32)

        for c in range(img.shape[-1]):
            band = img[..., c]
            min_val = band.min()
            max_val = band.max()
            if max_val - min_val > 0:
                img[..., c] = (band - min_val) / (max_val - min_val)
            else:
                img[..., c] = 0.0

        # ---- BINARIZE MASK ----
        mask = (mask > 0).astype(np.float32)

        # ---- ADD CHANNEL DIM TO MASK ----
        mask = np.expand_dims(mask, axis=-1)  # (H, W, 1)

        images.append(img)
        masks.append(mask)

    if len(images) == 0:
        print("No valid image-mask pairs found.")
        return None, None

    images = np.array(images)
    masks = np.array(masks)

    print("Loaded data:")
    print("Images shape:", images.shape)
    print("Masks shape:", masks.shape)

    return images, masks

In [10]:
import os
import numpy as np
import rasterio
import tifffile as tiff
from sklearn.metrics import precision_score, recall_score, f1_score

def dice_coefficient(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def iou(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    union = np.sum(y_true_f) + np.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def save_image(image, filepath):
    # Remove channel if single channel
    if len(image.shape) == 3 and image.shape[-1] == 1:
        image = image.squeeze(-1)

    # Clip to [0,1]
    image = np.clip(image, 0, 1)

    image = (image * 255).astype(np.uint8)

    tiff.imwrite(filepath, image)

# def predict1(model, mask_dir, image_dir, output_dir, number=-1):

#     dice_scores = []
#     iou_scores = []
#     precisions = []
#     recalls = []
#     f1_scores = []

#     os.makedirs(output_dir, exist_ok=True)

#     # Use same filenames as images
#     image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.tif')])

#     count = 0

#     for file in image_files:
#         image_path = os.path.join(image_dir, file)
#         mask_path = os.path.join(mask_dir, file)

#         if not os.path.exists(mask_path):
#             print(f"Mask missing for {file}")
#             continue

#         # ---- READ IMAGE (MULTI-BAND) ----
#         with rasterio.open(image_path) as src:
#             img = src.read()  # (C, H, W)
#             img = np.transpose(img, (1, 2, 0))  # (H, W, C)

#         # ---- NORMALIZE (same as training) ----
#         img = img.astype(np.float32)
        # for c in range(img.shape[-1]):
        #     band = img[..., c]
        #     min_val = band.min()
        #     max_val = band.max()
        #     if max_val - min_val > 0:
        #         img[..., c] = (band - min_val) / (max_val - min_val)
        #     else:
        #         img[..., c] = 0.0
        # img = img.astype(np.float32) / 255.0

        # # ---- READ MASK ----
        # with rasterio.open(mask_path) as src:
        #     mask = src.read(1)

        # mask = (mask > 0).astype(np.uint8)

        # # ---- PREDICTION ----
        # predicted_mask = model.predict(np.expand_dims(img, axis=0))[0]

        # if count < 5:
        #     print(f"{file} → min: {predicted_mask.min():.4f}, max: {predicted_mask.max():.4f}")

        # # ---- THRESHOLD ----
        # predicted_mask_thresh = (predicted_mask > 0.15).astype(np.uint8)

        # # ---- SAVE ----
        # save_image(predicted_mask, os.path.join(output_dir, f"prob_{file}"))
        # save_image(predicted_mask_thresh, os.path.join(output_dir, f"pred_{file}"))

        # # ---- METRICS ----
        # dice = dice_coefficient(mask, predicted_mask_thresh)
        # iou_score = iou(mask, predicted_mask_thresh)

        # precision = precision_score(mask.flatten(), predicted_mask_thresh.flatten(), zero_division=0)
    #     recall = recall_score(mask.flatten(), predicted_mask_thresh.flatten(), zero_division=0)
    #     f1 = f1_score(mask.flatten(), predicted_mask_thresh.flatten(), zero_division=0)

    #     dice_scores.append(dice)
    #     iou_scores.append(iou_score)
    #     precisions.append(precision)
    #     recalls.append(recall)
    #     f1_scores.append(f1)

    #     count += 1
    #     if number != -1 and count >= number:
    #         break

    # # ---- AVERAGE METRICS ----
    # mean_dice = np.mean(dice_scores)
    # mean_iou = np.mean(iou_scores)
    # mean_precision = np.mean(precisions)
    # mean_recall = np.mean(recalls)
    # mean_f1 = np.mean(f1_scores)

    # print(f"Mean Dice Coefficient: {mean_dice}")
    # print(f"Mean IoU: {mean_iou}")
    # print(f"Mean Precision: {mean_precision}")
    # print(f"Mean Recall: {mean_recall}")
    # print(f"Mean F1 Score: {mean_f1}")

    # return mean_dice, mean_iou, mean_precision, mean_recall, mean_f1

def evaluate_and_save(model, X_test, y_test, output_dir, threshold=0.15):
    
    os.makedirs(output_dir, exist_ok=True)

    preds = model.predict(X_test, batch_size=1)

    dice_scores = []
    iou_scores = []
    precision_list = []
    recall_list = []
    f1_list = []

    for i in range(len(preds)):
        pred = preds[i].squeeze()      # (512, 512)
        gt = y_test[i].squeeze()       # (512, 512)

        # ---- DEBUG (optional) ----
        if i < 5:
            print(f"{i} → min: {pred.min():.4f}, max: {pred.max():.4f}")

        # ---- THRESHOLD ----
        pred_bin = (pred > threshold).astype(np.uint8)

        # ---- SAVE IMAGES ----
        # probability map
        cv2.imwrite(os.path.join(output_dir, f"prob_{i}.png"), (pred * 255).astype(np.uint8))
        
        # predicted mask
        cv2.imwrite(os.path.join(output_dir, f"pred_{i}.png"), pred_bin * 255)
        
        # ground truth
        cv2.imwrite(os.path.join(output_dir, f"gt_{i}.png"), gt * 255)

        # ---- METRICS ----
        pred_flat = pred_bin.flatten()
        gt_flat = gt.flatten()

        intersection = np.sum(gt_flat * pred_flat)

        dice = (2 * intersection + 1e-6) / (
            np.sum(gt_flat) + np.sum(pred_flat) + 1e-6
        )

        iou = (intersection + 1e-6) / (
            np.sum(gt_flat) + np.sum(pred_flat) - intersection + 1e-6
        )

        precision = precision_score(gt_flat, pred_flat, zero_division=1)
        recall = recall_score(gt_flat, pred_flat, zero_division=1)
        f1 = f1_score(gt_flat, pred_flat, zero_division=1)

        dice_scores.append(dice)
        iou_scores.append(iou)
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)

    # ---- FINAL METRICS ----
    print("\n===== FINAL RESULTS =====")
    print("Mean Dice:", np.mean(dice_scores))
    print("Mean IoU:", np.mean(iou_scores))
    print("Mean Precision:", np.mean(precision_list))
    print("Mean Recall:", np.mean(recall_list))
    print("Mean F1:", np.mean(f1_list))

    return (
        np.mean(dice_scores),
        np.mean(iou_scores),
        np.mean(precision_list),
        np.mean(recall_list),
        np.mean(f1_list)
    )


In [16]:
from sklearn.model_selection import train_test_split
import os
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

model_types = ['residual']   # or ['residual'], ['dense']

image_dir = '/kaggle/input/datasets/vissshhhwa/coastline-dataset/dataset/images'
mask_dir  = '/kaggle/input/datasets/vissshhhwa/coastline-dataset/dataset/masks'

train = True

# ---- LOAD DATA ----
images, masks = load_data_new(image_dir, mask_dir)

print(images.shape)
print(masks.shape)

# ---- SPLIT DATA ----
X_train, X_temp, y_train, y_temp = train_test_split(
    images, masks, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

input_shape = X_train.shape[1:]
print(y_train.shape)
print(y_val.shape)

import tensorflow as tf

def dice_loss(y_true, y_pred):
    smooth = 1e-6

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # flatten everything (SAFE + BEST)
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true_f * y_pred_f)

    return 1 - (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )
    
def dice_metric(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.3, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2. * intersection + 1e-6) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + 1e-6
    )

def weighted_bce(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # compute BCE manually (no shape issues)
    bce = -(y_true * tf.math.log(y_pred + 1e-7) +
            (1 - y_true) * tf.math.log(1 - y_pred + 1e-7))

    weights = y_true * 5.0 + 1.0

    return tf.reduce_mean(bce * weights)
    
def combined_loss(y_true, y_pred):
    return weighted_bce(y_true, y_pred) + 0.5 * dice_loss(y_true, y_pred)

for model_type in model_types:

    model_name = f'UNet_{model_type}'
    print(model_name)

    # ---- SELECT MODEL ----
    if model_type == 'dense':
        model_fn = dense_unet

    elif model_type == 'residual':
        model_fn = residual_unet

    elif model_type == 'unet++':
        model_fn = lambda shape: dense_unet_plus_plus(shape, base_filters=32)

    # ---- TRAIN ----
    if train:
        model = model_fn(input_shape)

        model.compile(
            optimizer='adam',
            loss=combined_loss,
            metrics=[dice_metric]
        )

        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        )

        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=20,
            batch_size=2,
            callbacks=[early_stopping]
        )

    #     model.save(model_name)

    # # ---- LOAD MODEL ----
    # model = tf.keras.models.load_model(model_name, compile=False)
    # model.compile()

    # # ---- TEST EVALUATION ----
    # output_dir = f'./output/{model_type}/'
    # os.makedirs(output_dir, exist_ok=True)

    # print("\nRunning on TEST set...")

    # # Save test data temporarily for predict1
    # test_img_dir = "./temp_test_images"
    # test_mask_dir = "./temp_test_masks"

    # os.makedirs(test_img_dir, exist_ok=True)
    # os.makedirs(test_mask_dir, exist_ok=True)

    # import tifffile as tiff

    # # Save test tiles to temp folders
    # for i in range(len(X_test)):
    #     tiff.imwrite(os.path.join(test_img_dir, f"{i}.tif"), X_test[i])
    #     tiff.imwrite(os.path.join(test_mask_dir, f"{i}.tif"), y_test[i].squeeze())

    # # ---- PREDICT ----
    # predict1(
    #     model,
    #     test_mask_dir,
    #     test_img_dir,
    #     output_dir
    # )

Loaded data:
Images shape: (348, 512, 512, 3)
Masks shape: (348, 512, 512, 1)
(348, 512, 512, 3)
(348, 512, 512, 1)
Train: (243, 512, 512, 3)
Val: (52, 512, 512, 3)
Test: (53, 512, 512, 3)
(243, 512, 512, 1)
(52, 512, 512, 1)
UNet_residual
Epoch 1/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 123s 763ms/step - dice_metric: 0.1210 - loss: 1.5185 - val_dice_metric: 0.0429 - val_loss: 1.3432
Epoch 2/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 76s 625ms/step - dice_metric: 0.1219 - loss: 1.4322 - val_dice_metric: 0.0429 - val_loss: 1.1078
Epoch 3/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 76s 619ms/step - dice_metric: 0.1143 - loss: 1.4759 - val_dice_metric: 0.0429 - val_loss: 1.1451
Epoch 4/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 75s 616ms/step - dice_metric: 0.1120 - loss: 1.3646 - val_dice_metric: 0.0429 - val_loss: 1.1177
Epoch 5/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 76s 620ms/step - dice_metric: 0.1233 - loss: 1.3784 - val_dice_metric: 0.0428 - val_loss: 1.1150
Epoch 6/20
122/122 ━━━━━━━━━━━━━━━━━━━━ 76s 623ms/step - dice_metric: 0.10

In [17]:

model_path = "/kaggle/working/UNet_residual.keras"
model.save(model_path)

# ---- LOAD MODEL ----
model = tf.keras.models.load_model(model_path, compile=False)

# Recompile with same loss if needed (optional but safe)
model.compile()

# ---- OUTPUT DIRECTORY ----
output_dir = "/kaggle/working/output/"
os.makedirs(output_dir, exist_ok=True)

for t in [0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]:
    print(f"\nThreshold = {t}")
    results = evaluate_and_save(model, X_test, y_test, output_dir, threshold=t)

    print("\nFinal Results:", results)

# # ---- RUN EVALUATION + SAVE OUTPUTS ----
# results = evaluate_and_save(
#     model,
#     X_test,
#     y_test,
#     output_dir,
#     threshold=0.15   # you can tune this later
# )

# print("\nFinal Results:", results)


Threshold = 0.15
53/53 ━━━━━━━━━━━━━━━━━━━━ 7s 95ms/step
0 → min: 0.0198, max: 0.6742
1 → min: 0.0138, max: 0.5525
2 → min: 0.0251, max: 0.5890
3 → min: 0.0270, max: 0.5888
4 → min: 0.0280, max: 0.5666

===== FINAL RESULTS =====
Mean Dice: 0.11618201462030671
Mean IoU: 0.09982567539567326
Mean Precision: 0.10056989160163059
Mean Recall: 0.9974981498454067
Mean F1: 0.11618201461751956

Final Results: (np.float64(0.11618201462030671), np.float64(0.09982567539567326), np.float64(0.10056989160163059), np.float64(0.9974981498454067), np.float64(0.11618201461751956))

Threshold = 0.2
53/53 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step
0 → min: 0.0198, max: 0.6742
1 → min: 0.0138, max: 0.5525
2 → min: 0.0251, max: 0.5890
3 → min: 0.0270, max: 0.5888
4 → min: 0.0280, max: 0.5666

===== FINAL RESULTS =====
Mean Dice: 0.11595578374893167
Mean IoU: 0.0994130739908847
Mean Precision: 0.10066502644738352
Mean Recall: 0.9963178583731873
Mean F1: 0.11595578374612413

Final Results: (np.float64(0.115955783748931